In [5]:
#  The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
#import sendgrid
#from sendgrid.helpers.mail import Mail, Email, To, Content
#import mailersend
#from mailersend import Email
import os
import asyncio
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException


In [6]:
# The usual starting point

load_dotenv(override=True)


True

In [9]:
# Let's just check emails are working for you

def send_test_email():
    # 1. Setup Configuration
    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key['api-key'] = os.environ.get('BREVO_API_KEY')

    # 2. Initialize API instance
    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(sib_api_v3_sdk.ApiClient(configuration))

    # 3. Define the email contents
    sender = {"name": "Rocksteady Analytics", "email": "rockstedyanalytics@gmail.com"}
    to = [{"email": "rockstedyanalytics@gmail.com"}]
    subject = "Test email"
    html_content = "<html><body><h1>This is an important test email</h1></body></html>"
    
    # 4. Create the SendSmtpEmail object
    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        to=to, 
        html_content=html_content, 
        sender=sender, 
        subject=subject
    )

    try:
        # 5. Send the email
        api_response = api_instance.send_transac_email(send_smtp_email)
        print(f"Email sent successfully! Message ID: {api_response.message_id}")
    except ApiException as e:
        print(f"Exception when calling TransactionalEmailsApi->send_transac_email: {e}")

send_test_email()

Email sent successfully! Message ID: <202603270134.76142053465@smtp-relay.mailin.fr>


In [10]:
# Give intsructions

instructions1 = (
    "You are a professional sales agent representing Nate Vavrock of Rockstedy Analytics, "
    "a freelance data analytics and automation service that builds custom dashboards, data platforms, "
    "and AI-driven reporting tools for small businesses. "
    "You write professional, confident cold emails that convey credibility and technical expertise."
)

instructions2 = (
    "You are a witty, engaging sales agent representing Rockstedy Analytics, "
    "a service that helps small businesses automate their data workflows and visualize real insights "
    "through custom dashboards and AI analytics. "
    "You write humorous, conversational cold emails that grab attention and get replies."
)

instructions3 = (
    "You are a no-nonsense sales agent representing Rockstedy Analytics, "
    "a data analytics and automation service that builds dashboards, integrates APIs, and delivers results fast. "
    "You write concise, straight-to-the-point cold emails focused on value and efficiency."
)



In [11]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [12]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Unlock Data-Driven Success for Your Business

Hi [Recipient's Name],

I hope this message finds you well. My name is Nate Vavrock, and I'm reaching out from Rockstedy Analytics. We specialize in providing tailored data analytics and automation services designed specifically for small businesses like yours.

In today’s fast-paced market, making informed decisions is crucial. Our custom dashboards, data platforms, and AI-driven reporting tools empower businesses to visualize their data effectively and draw actionable insights—helping them save time and resources while increasing profitability.

Here’s how we can make a difference for you:

- **Tailored Dashboards**: We design user-friendly dashboards that bring your data to life, allowing you to track key metrics at a glance.
  
- **Automated Reporting**: Say goodbye to manual reporting. Our automation solutions ensure you receive timely, accurate data without the hassle.
  
- **AI-Driven Insights**: Leverage advanced analytics 

In [13]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Transform Your Data into Actionable Insights

Hi [Recipient's Name],

I hope this message finds you well. My name is Nate Vavrock, and I’m the founder of Rockstedy Analytics, where we specialize in providing tailored data analytics and automation solutions for small businesses.

In today’s data-driven world, turning raw data into actionable insights is crucial for staying ahead of the competition. We help businesses like yours develop custom dashboards, data platforms, and AI-driven reporting tools that not only streamline operations but also enhance decision-making.

Imagine having real-time insights at your fingertips—allowing your team to focus on what matters most while we handle the complexities of your data. 

I would love to explore how Rockstedy Analytics can support your business’s goals and help you unlock the full potential of your data. Are you available for a brief call this week to discuss your current analytics needs?

Looking forward to connecting!

Best regard

In [14]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [15]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")



Best sales email:
Subject: Let’s Turn Your Data Drama into a Success Saga! 🎭🚀

Hey [Recipient's Name],

I hope this email finds you in a good mood, preferably sipping coffee and not wrestling with spreadsheets! ☕️✋

Let’s be honest—managing your data shouldn’t feel like you’re trying to solve a Rubik's Cube blindfolded, right? (And don’t even get me started on the headaches of manual reports!) 

At Rockstedy Analytics, we specialize in transforming your data chaos into clarity, turning pesky numbers into beautiful dashboards that even your grandma could understand. No more digging through mountains of data to find those golden nuggets of insight! 💎✨

Imagine automating your data workflows and getting real insights delivered to you faster than you can say, “Why is my spreadsheet on fire?!” 🔥 With our custom dashboards and AI analytics, you’ll not only save time but also impress your team (and maybe even yourself) with the clarity of your data-driven decisions.

Curious? Let’s chat! I pr

Now go and check out the trace:

https://platform.openai.com/traces

In [17]:
print(sales_agent1)


Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a professional sales agent representing Nate Vavrock of Rockstedy Analytics, a freelance data analytics and automation service that builds custom dashboards, data platforms, and AI-driven reporting tools for small businesses. You write professional, confident cold emails that convey credibility and technical expertise.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choic

In [20]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    
    # 1. Setup Brevo Configuration
    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key['api-key'] = os.environ.get('BREVO_API_KEY')

    # 2. Initialize the API instance
    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(sib_api_v3_sdk.ApiClient(configuration))

    # 3. Define the email payload
    # Note: 'to' must be a list of dictionaries
    sender = {"name": "Rocksteady Analytics", "email": "rockstedyanalytics@gmail.com"}
    to = [{"email": "rockstedyanalytics@gmail.com"}]
    subject = "Sales email"
    
    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        to=to,
        sender=sender,
        subject=subject,
        html_content=f"<html><body>{body}</body></html>"
    )

    try:
        # 4. Send the email
        api_instance.send_transac_email(send_smtp_email)
        return {"status": "success"}
    except ApiException as e:
        return {"status": "error", "message": str(e)}

In [21]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000026A9ABCECA0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)